**Programmer: python_scripts (Abhijith Warrier)**

**PYTHON SCRIPT TO *FIND THE FIRST AND PAST-THE-LAST POSITIONS OF REPEATED VALUES USING LOWER AND UPPER BOUNDS. 🐍🔍***

Finding one matching value does not tell us where a repeated run starts or ends. Lower and upper bounds turn binary search into a search for boundaries.

We’ll implement both searches, trace their decisions on a sorted list, count duplicates, and verify missing-value and edge cases against Python’s `bisect` module. The input must remain sorted in ascending order and unchanged during each search.

---

### 1. Imports

Python standard library only. No package installation is required.

In [1]:
from bisect import bisect_left, bisect_right
from itertools import combinations_with_replacement
from pathlib import Path
import json
import platform


---

### 2. Search for a boundary

Both searches keep a half-open unclassified interval `[lo, hi)`, beginning with `lo = 0` and `hi = len(a)`. The answer is a position from `0` through `len(a)`, inclusive.

- Lower bound: first position with value **>= target**. Values before `lo` are `< target`; values from `hi` onward are `>= target`.
- Upper bound: first position with value **> target**. Values before `lo` are `<= target`; values from `hi` onward are `> target`.

On equality, lower bound moves `hi` left to `mid`; upper bound moves `lo` right past `mid`. Neither returns immediately on a match. When `lo == hi`, the boundary is known.

In [2]:
def lower_bound(a, target):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo


def upper_bound(a, target):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] <= target:
            lo = mid + 1
        else:
            hi = mid
    return lo


---

### 3. Trace the repeated values

For `[2, 4, 4, 4, 7, 9]` and target `4`, the boundaries enclose indices `[1, 4)`. The last matching index is `upper - 1` **only when the range is nonempty**. The upper bound itself is the position after the matches, not the last match.

The separate trace helper records decisions for the animation; the two search functions above do not allocate a trace.

In [3]:
def trace_bound(a, target, upper=False):
    lo, hi = 0, len(a)
    steps = []
    while lo < hi:
        old_lo, old_hi = lo, hi
        mid = (lo + hi) // 2
        move_right = a[mid] <= target if upper else a[mid] < target
        if move_right:
            lo = mid + 1
        else:
            hi = mid
        assert hi - lo < old_hi - old_lo
        assert all(v <= target if upper else v < target for v in a[:lo])
        assert all(v > target if upper else v >= target for v in a[hi:])
        steps.append({"lo": old_lo, "hi": old_hi, "mid": mid,
                      "value": a[mid], "next_lo": lo, "next_hi": hi})
    return lo, steps

values = [2, 4, 4, 4, 7, 9]
target = 4
left, right = lower_bound(values, target), upper_bound(values, target)
assert (left, right) == (1, 4)
assert values[left:right] == [4, 4, 4]
traces = {}
for name, upper in (("lower", False), ("upper", True)):
    answer, steps = trace_bound(values, target, upper)
    traces[name] = steps
    assert answer == (right if upper else left)
    print(name, "midpoints:", [s["mid"] for s in steps], "boundary:", answer)
print("Matching interval:", (left, right), "count:", right - left)


lower midpoints: [3, 1, 0] boundary: 1
upper midpoints: [3, 5, 4] boundary: 4
Matching interval: (1, 4) count: 3


---

### 4. Verify missing values and boundaries

If the target is absent, both searches return the same insertion position, so `upper - lower == 0`. An empty list gives `0`; a target above every value gives `len(a)`. Never index that past-the-end position.

We compare all sorted lists of lengths 0 through 6 over `{0, 1, 2, 3}`, for targets -1 through 4, against `bisect_left` / `bisect_right` and linear partition/count checks. This includes empty, all-equal, absent and edge targets.

In [4]:
examples = []
for a, x in ((values, 5), (values, 10), ([], 4), ([4, 4, 4], 4)):
    low, high = lower_bound(a, x), upper_bound(a, x)
    examples.append({"values": a, "target": x, "lower": low, "upper": high,
                     "count": high - low})
    print(examples[-1])

checked = 0
for length in range(7):
    for a in combinations_with_replacement(range(4), length):
        for x in range(-1, 5):
            low, high = lower_bound(a, x), upper_bound(a, x)
            assert (low, high) == (bisect_left(a, x), bisect_right(a, x))
            assert 0 <= low <= high <= len(a)
            assert all(v < x for v in a[:low])
            assert all(v >= x for v in a[low:])
            assert all(v <= x for v in a[:high])
            assert all(v > x for v in a[high:])
            assert high - low == a.count(x)
            for upper, expected in ((False, low), (True, high)):
                answer, _ = trace_bound(a, x, upper)
                assert answer == expected
            checked += 1
assert checked == 1260
print("Verified", checked, "array/target cases.")


{'values': [2, 4, 4, 4, 7, 9], 'target': 5, 'lower': 4, 'upper': 4, 'count': 0}
{'values': [2, 4, 4, 4, 7, 9], 'target': 10, 'lower': 6, 'upper': 6, 'count': 0}
{'values': [], 'target': 4, 'lower': 0, 'upper': 0, 'count': 0}
{'values': [4, 4, 4], 'target': 4, 'lower': 0, 'upper': 3, 'count': 3}
Verified 1260 array/target cases.


---

### 5. Save results and use the standard library

In application code, `bisect_left(a, target)` and `bisect_right(a, target)` provide these boundaries directly. Each search takes `O(log n)` time with constant-time indexing/comparisons and `O(1)` auxiliary space. Two searches remain `O(log n)`; counting by subtraction adds constant work.

Extracting `a[lower:upper]` copies the matching elements and costs `O(k)` for `k` matches. Sorting and inserting into a Python list have separate costs. Trace storage is `O(log n)` and its extra validation scans are test instrumentation, not part of the stated search complexity.

In [5]:
report = {"python": platform.python_version(), "values": values, "target": target,
          "lower": left, "upper": right, "count": right - left,
          "traces": traces, "edge_examples": examples, "verified_cases": checked}
Path("results.json").write_text(json.dumps(report, indent=2))
print("bisect result:", bisect_left(values, target), bisect_right(values, target))
print("Saved results.json")


bisect result: 1 4
Saved results.json


---

**Takeaways:** lower bound keeps equality on the right; upper bound moves past equality. Matches occupy `[lower, upper)` and their count is `upper - lower`. The returned positions are not a guarantee that the target exists.

---